<a href="https://colab.research.google.com/github/sita-aghasoy33/basic-llm-tutorials/blob/main/RAG_Chunking_with_MD_files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌳 RAG — PDF → Markdown + Recursive (token-əsaslı) Splitting

Bu notebook `RAG_PDF_to_Markdown.ipynb`-in davamıdır. Fərq yalnız **3-cü bölmədədir**:
sadə "başlığa görə böl, sonra sabit simvol uzunluğu ilə kəs" əvəzinə, burada **recursive character splitting** tətbiq olunur — həm də ölçü simvol yox, Hugging Face tokenizer-i ilə **token sayına** görə hesablanır.

Bütün model və kitabxanalar Hugging Face-dən lokal yüklənir — xarici API açarı lazım deyil, pulsuzdur.

**Qeyd:** GPU olmadan da işləyir, amma Colab-da `Runtime → Change runtime type → GPU` seçsəniz generasiya addımı sürətlənəcək.

Colab-da soldakı fayl panelindən `sirket_siyaseti.pdf` faylını yükləyin (və ya öz PDF-inizin adını dəyişin).

## 1. Kitabxanaların qurulması

In [1]:
!pip install -q pymupdf4llm sentence-transformers faiss-cpu transformers accelerate

import pymupdf4llm
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import re
from transformers import pipeline


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 53.2 MB/s eta 0:00:00


## 2. PDF-i Markdown-a çevirmək

`pymupdf4llm.to_markdown(...)` PDF-i başlıqlar, siyahılar və (mümkün qədər) cədvəllər qorunmuş şəkildə Markdown mətninə çevirir.

In [2]:
PDF_PATH = "/content/sirket_siyaseti_normal.pdf"  # öz PDF faylınızın yolu ilə əvəz edin

md_text = pymupdf4llm.to_markdown(PDF_PATH)

print(f"Ümumi simvol sayı: {len(md_text)}")
print(md_text[:500])


=== Document parser messages ===
Using Tesseract for OCR processing.
Ümumi simvol sayı: 1446
# **Uzaqdan İşləmə Siyasəti — Nordic Tech MMC** 

Sənəd nömrəsi: HR-POL-014  |  Təsdiq tarixi: 2026-01-15  |  Versiya: 2.1 

## **1. Məqsəd** 

Bu sənəd Nordic Tech MMC daxilində uzaqdan işləmə qaydalarını, işçilərin hüquq və vəzifələrini, həmçinin texniki və təhlükəsizlik tələblərini müəyyən edir. Siyasət bütün tam və yarım ştat işçilərinə şamil olunur. 

## **2. Uyğunluq şərtləri** 

Uzaqdan işləmək üçün işçi ən azı 6 ay şirkətdə çalışmış olmalı, birbaşa rəhbərin yazılı təsdiqini almalı və İT 


## 3. Embedding modelini əvvəlcədən yükləmək (tokenizer üçün lazımdır)

Bu dəfə embedding modelini chunk-lamadan **əvvəl** yükləyirik, çünki chunk ölçüsünü onun öz tokenizer-i ilə (simvol yox, **token sayı** ilə) hesablayacağıq. Bu vacibdir, çünki `all-MiniLM-L6-v2` modelinin max giriş limiti **256 token**-dir — token sayına görə bölsək, heç bir chunk kəsilib məlumat itirmir.

Bəzən bütün proses normal işləsə belə bəzi kiçik həcmli modellər promptu doğru emal edib cavab verə bilmir. Bunun səbəbi bəzən modelin öz gücü, əsasən də dil dəstəyi olur. Azərbaycan dili məhdud areala sahib dil olduğu üçün bir çox modellər, xüsusən kiçik modellər bu dil üzərində ya tamamilə təlim (train) olunmur, ya da olunmuş olsa belə, data həcminin nisbəti ingilis, rus, alman dili kimi dillərə nəzərən az olur. Bu da səbəb olur ki, model sualı oxusa, hətta vektor bazada həmin suala uyğun gələn kontekst chunklarını doğru tapıb çıxarsa belə, cavab vermir.

Bu embedding modelləri üçün də keçərlidir. Əgər biz Azərbaycan dilində işləyən və bu dildə mətnlərə əsasən cavab verən RAG qururuqsa, həm embedding (retrival mətni ədədi vektora çevirən) modelimiz, həm də response modellimiz (suala cavab verən) bu dili dəstəkləməlidir.

Bu səbəbdən biz embedding üçün daha sürətli işləyən `all-MiniLM-L6-v2` əvəzinə `paraphrase-multilingual-MiniLM-L12-v2` modelindən istifadə edirik. Bu model bir çox dili dəstəkləyir.



In [3]:
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def hf_token_length(text):
    """Mətnin HF tokenizer-inə görə token sayını qaytarır."""
    return len(embedding_model.tokenizer.encode(text, add_special_tokens=False))

# sürətli yoxlama
print(hf_token_length("Bu qısa bir cümlədir."))


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

8


## 4. Başlığa görə bölmək (əvvəlki addım)

Əvvəlki notebook-dakı kimi, əvvəlcə Markdown-ı başlıqlara (`#`, `##`, `###`) görə bölürük — hər bölmə öz mövzusunu saxlasın.

In [4]:
def split_by_headers(md_text):
    pattern = r"(?=^#{1,3} .+$)"
    sections = re.split(pattern, md_text, flags=re.MULTILINE)
    return [s.strip() for s in sections if s.strip()]

sections = split_by_headers(md_text)
print(f"{len(sections)} bölmə tapıldı")
sections

6 bölmə tapıldı


['# **Uzaqdan İşləmə Siyasəti — Nordic Tech MMC** \n\nSənəd nömrəsi: HR-POL-014  |  Təsdiq tarixi: 2026-01-15  |  Versiya: 2.1',
 '## **1. Məqsəd** \n\nBu sənəd Nordic Tech MMC daxilində uzaqdan işləmə qaydalarını, işçilərin hüquq və vəzifələrini, həmçinin texniki və təhlükəsizlik tələblərini müəyyən edir. Siyasət bütün tam və yarım ştat işçilərinə şamil olunur.',
 '## **2. Uyğunluq şərtləri** \n\nUzaqdan işləmək üçün işçi ən azı 6 ay şirkətdə çalışmış olmalı, birbaşa rəhbərin yazılı təsdiqini almalı və İT departamentinin tələb etdiyi təhlükəsizlik təlimatlarını keçməlidir. Sınaq müddətində olan işçilər bu imkandan istifadə edə bilməzlər.',
 '## **3. İş saatları və əlçatanlıq** \n\nUzaqdan işləyən əməkdaşlar əsas iş saatları ərzində (10:00–16:00, Bakı vaxtı ilə) komanda kommunikasiya vasitələrində əlçatan olmalıdırlar. Həftəlik iş saatı normal ofis rejimi ilə eynidir — 40 saat.',
 '## **4. Təhlükəsizlik tələbləri** \n\nBütün uzaqdan iş üçün istifadə olunan cihazlarda disk şifrələməsi, 

## 5. Recursive character splitting (token-əsaslı)

Hər bölmə hələ də çox uzun ola bilər (məs. bir başlıq altında 3 səhifəlik mətn). Bunun üçün **recursive splitting** tətbiq edirik:

1. Əvvəlcə ən "təbii" ayırıcı ilə bölməyə çalış: `"\n\n"` (paraqraf).
2. Nəticə hələ də `max_tokens`-dan böyükdürsə, növbəti ayırıcı ilə təkrar böl: `"\n"` (sətir).
3. Sonra `". "` (cümlə), sonra `" "` (söz) — hər addımda daha "kiçik" sərhədə enir.
4. Heç bir ayırıcı kömək etmirsə (məs. çox uzun tək söz/token axını), son çarə kimi sabit uzunluqla kəsir.
5. Kiçik hissələri isə `max_tokens`-a çatana qədər overlap ilə yenidən birləşdirir ki, çox sayda mini-chunk yaranmasın.

Bu, LangChain-in `RecursiveCharacterTextSplitter`-inin sadələşdirilmiş, öz əlimizlə yazılmış versiyasıdır.

In [5]:
SEPARATORS = ["\n\n", "\n", ". ", " "]

def split_on_separator(text, separator):
    if separator == "":
        return list(text)
    parts = text.split(separator)
    # ayırıcını itirməyək (cümlə/paraqraf bütövlüyü üçün)
    return [p + separator if i < len(parts) - 1 else p for i, p in enumerate(parts)]


def recursive_split(text, separators, max_tokens, length_fn):
    """Mətni verilmiş ayırıcılar siyahısı ilə rekursiv şəkildə bölür."""
    if length_fn(text) <= max_tokens:
        return [text]

    if not separators:
        # son çarə: sabit simvol uzunluğu ilə kəs
        mid = len(text) // 2
        return recursive_split(text[:mid], [], max_tokens, length_fn) + \
               recursive_split(text[mid:], [], max_tokens, length_fn)

    sep, rest_separators = separators[0], separators[1:]
    pieces = split_on_separator(text, sep)

    results = []
    for piece in pieces:
        if length_fn(piece) <= max_tokens:
            results.append(piece)
        else:
            # bu hissə hələ də böyükdür -> növbəti ayırıcı ilə davam et
            results.extend(recursive_split(piece, rest_separators, max_tokens, length_fn))
    return results


def merge_with_overlap(pieces, max_tokens, overlap_tokens, length_fn):
    """Kiçik hissələri max_tokens-a çatana qədər birləşdirir, sonuncu overlap qədərini növbəti chunk-a köçürür."""
    chunks = []
    current = ""

    for piece in pieces:
        candidate = current + piece
        if length_fn(candidate) <= max_tokens:
            current = candidate
        else:
            if current.strip():
                chunks.append(current.strip())
            # overlap: əvvəlki chunk-ın son hissəsini yeni chunk-a "daşı"
            overlap_text = current[-overlap_tokens * 4:]  # təxmini simvol qarşılığı
            current = overlap_text + piece

    if current.strip():
        chunks.append(current.strip())
    return chunks


def build_chunks(sections, max_tokens=200, overlap_tokens=30):
    all_chunks = []
    for section in sections:
        pieces = recursive_split(section, SEPARATORS, max_tokens, hf_token_length)
        merged = merge_with_overlap(pieces, max_tokens, overlap_tokens, hf_token_length)
        all_chunks.extend(merged)
    return all_chunks


chunks = build_chunks(sections, max_tokens=200, overlap_tokens=30)

print(f"{len(sections)} bölmə -> {len(chunks)} chunk (token-əsaslı recursive split)")
print("\nChunk uzunluqları (token):", [hf_token_length(c) for c in chunks[:10]], "...")
print("\nNümunə chunk:\n", chunks[0][:400])


6 bölmə -> 6 chunk (token-əsaslı recursive split)

Chunk uzunluqları (token): [43, 55, 64, 64, 71, 60] ...

Nümunə chunk:
 # **Uzaqdan İşləmə Siyasəti — Nordic Tech MMC** 

Sənəd nömrəsi: HR-POL-014  |  Təsdiq tarixi: 2026-01-15  |  Versiya: 2.1


## 6. FAISS vektor indeksi

Yuxarıda embedding modelini formalaşdırdıq. Lakin hələ istifadə etmədik. Çünki ilk olaraq mətnimizi kiçik hissələrə bölməliyik. Bu RAG sisteminin əsasıdır. Mətnlər məntiqli olaraq bölünməlidir ki, istifadəçi sorğusuna əsasən bot gedib mətn hissələrini axtaranda cavab olaraq əlaqəsiz məlumat verməsin.

Təsəvvür edin ki, mən şirkətin daxili siyasət sənədini hissələrə bölərkən əməkhaqqı siyasəti ilə, uzaqdan işləmə siyasəti haqqındakı məlumatı eyni chunka salmışam. Bu zaman istifadəçi uzaqdan işlə bağlı soruşanda, bot ona əməkhaqqı haqqında da məlumat verəcək. Çünki mətnlər istifadəçinin sualda istifadə etdiyi sözlərlə məna yaxınlığına əsasən axtarılır. Və sonra botun cavabı həmin mətnin tərkibinə əsasən formalaşır. Mətndə nə məlumat varsa, o cavabda da öz əksini tapacaq. Unutmayaq ki, ai botlar cavabı ehtimal əsasında formalaşdırır. Və "başa düşmək" anlayışı onlar üçün sadəcə kontekstə uyğun istifadəçinin sualına cavab verməkdir.

Biz pdf faylımızı md (markdown) faylına çevirdik. Bu zaman **bold**, *italic*, və ya header kimi formatlar "*", "#" kimi işarələrə çevrilir. Nəticədə daha mürəkkəb yanaşmaya ehtiyac qalmadan başlıqları tapmaqla, mətni mənalı hissələrə bölmək olur. Əgər bu mümkün olmasa, mətnin məna bağlılığını özü taparaq hissələrə bölən üçüncü bir modeldən istifadə edəcəkdik.

Aşağıda `split_by_headers()` funksiyası ilə böldüyümüz sectionları vektorlara çevirir və indeksləyirik. Bu funksiya "md" formatındakı mətni "#" işarəsinə əsasən başlıqlara bölür. Daha sonra indeksləmə əməliyyatı sayəsində hər chunka sıra nömrəsi verilir. Bunun sayəsində model suala uyğun hissəni tapanda bizə onun indeksini qaytaracaq.

In [6]:
chunk_embeddings = embedding_model.encode(sections, convert_to_numpy=True, show_progress_bar=True)
print("Embedding ölçüsü:", chunk_embeddings.shape)

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)
print(f"İndeksdə {index.ntotal} chunk var, hər biri {dimension} ölçülü vektor")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding ölçüsü: (6, 384)
İndeksdə 6 chunk var, hər biri 384 ölçülü vektor


## 7. Retrieval funksiyası


Daha sonra final botumuzu hazırladıqda bizə lazım olacaq `retiriever()` funksiyası yaradırıq. Bu funksiya sayəsində bot istifadəçinin verdiyi suala uyğun verilmiş "k" sayda chunkı çəkib gətirəcək. Sonra bot bu məhz bu seçilmiş mətnləri promptun tərkib hissəsinə çevirərək qəbul edib sualı cavablandıracaq.

Burada k-nı varsayılan olaraq 3 götürmüşük. Adətən 2, yaxud 3 olması məqsədəuyğundur. 1 çox aşağıdır və suala uyğun ancaq 1 mətni çıxararaq əhatəli cavab verməyə bilər, k-nı daha böyük seçdikdə isə suala daha az uyğun olan mətnləri də prompta daxil edəcək.

In [7]:
def retrieve(query, k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

# sürətli yoxlama
retrieve("sənədin əsas mövzusu nədir?")


['## **2. Uyğunluq şərtləri** \n\nUzaqdan işləmək üçün işçi ən azı 6 ay şirkətdə çalışmış olmalı, birbaşa rəhbərin yazılı təsdiqini almalı və İT departamentinin tələb etdiyi təhlükəsizlik təlimatlarını keçməlidir. Sınaq müddətində olan işçilər bu imkandan istifadə edə bilməzlər.',
 '## **1. Məqsəd** \n\nBu sənəd Nordic Tech MMC daxilində uzaqdan işləmə qaydalarını, işçilərin hüquq və vəzifələrini, həmçinin texniki və təhlükəsizlik tələblərini müəyyən edir. Siyasət bütün tam və yarım ştat işçilərinə şamil olunur.',
 '## **5. Avadanlıq və xərclər** \n\nŞirkət uzaqdan işləyən əməkdaşlara noutbuk və zəruri proqram təminatı təqdim edir. İnternet xərcinin bir hissəsi (aylıq 30 AZN-ə qədər) əsaslandırıcı qəbz təqdim edildiyi təqdirdə şirkət tərəfindən ödənilir.']

## 8. Generasiya modeli

`flan-t5-base` — kiçikdir (~250M parametr), instruksiya izləməyə uyğundur və CPU-da da rahat işləyir. Daha güclü cavab üçün `Qwen/Qwen2.5-1.5B-Instruct` kimi modelə keçmək olar (GPU tövsiyə olunur).

In [8]:
!pip install -q -U "bitsandbytes>=0.46.1"
!pip show bitsandbytes | grep Version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.9 MB/s eta 0:00:00
Version: 0.50.2


In [9]:
from transformers import BitsAndBytesConfig
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    device_map="auto",
    model_kwargs={"quantization_config": BitsAndBytesConfig(load_in_4bit=True)},
)
print(generator.model.device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

cuda:0


## Prompt dizaynı — limitations və format

Əvvəlki notebook-larla eyni prinsip: modelə yalnız kontekstdən istifadə etməyi, kontekstdə cavab yoxdursa bunu bildirməyi və qısa/aydın formatda cavab verməyi tapşırırıq.

## 9. RAG pipeline — limitations və format daxil olan prompt

Prompt mühəndisliyi RAG pipeline-lar üçün də ciddi rol oynayır. Burada da rol, limitasiya, format, kontekst anlayışları mövcuddur. Bizim üçün kontekst rolunu burada retriever funksiyası ilə suala ən uyğun seçilmiş k sayda mətnlərin birləşməsi oynayacaq.

Yekun olaraq modelə daxil olan prompt sistem instruksiyası, suala uyğun k sayda mətn chunkları və istifadəçinin göndərdiyi sualın cəmi olacaq.

`build_messages()` funksiyası bu saydığımız materialları `dictionary` formatında olan prompt tipinə çevirir.

`rag_answer()` funksiyası isə əldə olunan yekun prompt əsasında suala cavab qaytarmaq üçün yaradılıb.

Aşağıdakı hissə bu funksiyanın əsas hissəsidir. Bu hissə vasitəsilə `generator()` funksiyası `messages` parametri ilə ötürülən prompt vasitəsilə cavab generasiya edir. `repitition_penalty` burada sanki `temperature` anlayışınıb əksi kimidir: asistant botu təkrarçılıq edən cavablar qaytardığı üçün cəzalandırır.

```
generator(
        messages,
        max_new_tokens=200,
        do_sample=False,          # deterministik cavab
        repetition_penalty=1.1,   # eyni cümləni təkrarlamasın
    )
```

Aşağıdakı əməliyyat isə əldə olunan **output** nəticəsinin içindən ancaq botun cavabını seçib çıxarır:

```
result[0]["generated_text"][-1]["content"].strip()
```

In [10]:
SYSTEM_INSTRUCTIONS = """

Sən şirkət sənədləri üzrə köməkçisən. İstifadəçinin sualına yalnız verilən KONTEKST əsasında, Azərbaycan dilində cavab ver.

Qaydalar:
1. Yalnız KONTEKST-dəki məlumatdan istifadə et, öz biliyini əlavə etmə.
2. KONTEKST-də suala aid məlumat varsa (hətta qismən), mütləq onunla cavab ver.
3. Yalnız KONTEKST-də suala aid heç bir məlumat olmadıqda yaz: "Bu sualın cavabı verilən sənəddə yoxdur."
4. Cavab qısa və aydın olsun (lazım gələrsə bullet ilə). Uydurma fakt, rəqəm və ya tarix yazma.

"""

def build_messages(query, context):
    # Qwen chat modelidir -> prompt system/user mesajları şəklində verilməlidir
    return [
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": f"KONTEKST:\n{context}\n\nSUAL: {query}"},
    ]


def rag_answer(query, k=3):
    context_chunks = retrieve(query, k=k)
    context = "\n\n---\n\n".join(context_chunks)
    messages = build_messages(query, context)
    result = generator(
        messages,
        max_new_tokens=200,
        do_sample=False,          # deterministik cavab
        repetition_penalty=1.1,   # eyni cümləni təkrarlamasın
    )
    # chat formatında generated_text mesajlar siyahısıdır -> sonuncu = modelin cavabı
    answer = result[0]["generated_text"][-1]["content"].strip()
    return {"answer": answer, "sources": context_chunks, "messages": messages}

## 10. Test sualları

Diqqət: ikinci sual **qəsdən** sənədlə əlaqəsizdir. Bu modelin "bilmirəm" deyib-demədiyini yoxlamaq üçün nəzərdə tutulmuşdur.

Nəticə olaraq həm istifadəçinin sualını, həm suala uyğun tapılan chunkları, həm də botun cavabını görə bilərsiniz.

In [11]:
test_questions = [
    "Sənəddə əsas mövzu nədir?",
    "Ayın səthi nədən ibarətdir?",   # qəsdən sənədlə əlaqəsiz sual
    "Bu sənədə əsasən əsas qaydalardan/nöqtələrdən biri hansıdır?",
]

for q in test_questions:
    result = rag_answer(q, k=3)
    print(f"Sual: {q}")
    print(f"Cavab: {result['answer']}")
    print("Mənbələr:")
    for s in result['sources']:
        print("-", s[:150].replace("\n", " ") + "...")
    print("-" * 60)

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `

Sual: Sənəddə əsas mövzu nədir?
Cavab: Sənəddə əsas mövzu uzaqdan işləmə qaydaları, işçi hüquqları və təhlükəsizlik tələbləri haqqındadır.
Mənbələr:
- ## **2. Uyğunluq şərtləri**   Uzaqdan işləmək üçün işçi ən azı 6 ay şirkətdə çalışmış olmalı, birbaşa rəhbərin yazılı təsdiqini almalı və İT departame...
- ## **5. Avadanlıq və xərclər**   Şirkət uzaqdan işləyən əməkdaşlara noutbuk və zəruri proqram təminatı təqdim edir. İnternet xərcinin bir hissəsi (ayl...
- ## **1. Məqsəd**   Bu sənəd Nordic Tech MMC daxilində uzaqdan işləmə qaydalarını, işçilərin hüquq və vəzifələrini, həmçinin texniki və təhlükəsizlik t...
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sual: Ayın səthi nədən ibarətdir?
Cavab: Bu sualın cavabı verilən sənəddə yoxdur.
Mənbələr:
- ## **2. Uyğunluq şərtləri**   Uzaqdan işləmək üçün işçi ən azı 6 ay şirkətdə çalışmış olmalı, birbaşa rəhbərin yazılı təsdiqini almalı və İT departame...
- ## **5. Avadanlıq və xərclər**   Şirkət uzaqdan işləyən əməkdaşlara noutbuk və zəruri proqram təminatı təqdim edir. İnternet xərcinin bir hissəsi (ayl...
- ## **1. Məqsəd**   Bu sənəd Nordic Tech MMC daxilində uzaqdan işləmə qaydalarını, işçilərin hüquq və vəzifələrini, həmçinin texniki və təhlükəsizlik t...
------------------------------------------------------------
Sual: Bu sənədə əsasən əsas qaydalardan/nöqtələrdən biri hansıdır?
Cavab: - İşçi ən azı 6 ay şirkətdə çalışması tələbi var.
- Birbaşa rəhbərin yazılı təsdiqini almağı tələb edir.
- İT departamentından təhlükəsizlik təlimatlarını keçməyi tələb edir.
- Sınaq müddətində olan işçilər uzaqdan işləyə bilər.
- Şirkət uzaqdan işləyən əməkdaşlara noutbuku və zəruri proqram təminatını t

## 📝 Bu üsulun üstünlükləri (əvvəlki iki notebook ilə müqayisədə)

| Üsul | Sərhəd növü | Ölçü meyarı | Zəif tərəfi |
|---|---|---|---|
| `RAG_PDF_Direct.ipynb` | sabit simvol | simvol sayı | cümlələr ortadan kəsilir |
| `RAG_PDF_to_Markdown.ipynb` | yalnız başlıq | simvol sayı | uzun bölmələr yenə sabit kəsilir |
| **Bu notebook** | paraqraf → sətir → cümlə → söz (rekursiv) | **token sayı** (HF tokenizer) | ən mürəkkəb, bir az daha yavaş |

**Bonus tapşırıq:** `max_tokens` (məs. 100, 200, 300) və `overlap_tokens` dəyərlərini dəyişərək chunk sayının və retrieval keyfiyyətinin necə dəyişdiyini müşahidə edin.

In [12]:
rag_answer("Uzaqdan işləyə bilərəmmi", k=3)['answer']

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'- Şirkət uzaqdan işləmək üçün işçi ən azı 6 ay şirkətdə çalışmış olmalıdır.\n- Bu imkan sınaq müddətində olan işçilərlər üçün açılmamışdır.\n\nYani, uzaqdan işləyə bilərsiniz lakin sınaq müddətində olan işçilər bu imkanlardan istifadə edə bilməz.'